# Smart Semsar — furniture photo → 3D with a diffusion model (Hunyuan3D-2 mini) — Kaggle

صورة قطعة فرش واحدة → موديل 3D (GLB)، بموديل **diffusion** شغال على الـ GPU المجاني بتاع Kaggle (حوالي 30 ساعة في الأسبوع).

**The model:** [Hunyuan3D-2 mini](https://github.com/Tencent/Hunyuan3D-2) by Tencent. A diffusion transformer (flow matching)
denoises a 3D shape conditioned on the photo; Hunyuan3D-Paint (also diffusion) can then paint its colours.

**Kaggle setup (once)**
1. Right panel → *Session options*: **Accelerator = GPU T4 x2** (not P100: the current PyTorch no longer supports it), **Internet = On**
   (Internet needs a phone-verified Kaggle account).
2. *Add-ons → Secrets*: add `NGROK_TOKEN` (https://dashboard.ngrok.com/get-started/your-authtoken) and `SMARTSEMSAR_KEY`
   (any password you choose), and tick both for this notebook. Only needed for 5B.
3. For 5A: *Add Input → Upload* your furniture photos as a dataset; every photo under `/kaggle/input` is converted.
4. **Run All**. The install cell restarts the kernel once by itself; after that, **Run All** again.

**Two ways to use it**
- **A. GLB files:** each photo becomes a `.glb` in `/kaggle/working` (Output panel → download), for the app's *"…or a GLB model"*.
- **B. Server:** the same `/health` and `/generate` endpoints as `cv_service/colab_server.ipynb`: put the two printed lines in the
  project's `.env` (or in `COLAB_URL` of `smartsemsar_demo.ipynb` / `SmartSemsar_multi_photo.ipynb`).

Kaggle has about 29 GB of RAM (Colab free has 12), so `TEXTURE = True` (colours) may work here; it adds a compile step and a big download.

## 1. Settings and GPU check

In [ ]:
TEXTURE = True    # True = paint colours with Hunyuan3D-Paint (bigger download and a compile step; try it on Kaggle's RAM)
STEPS = 30         # diffusion steps: more = finer and slower (20 to 50)
OCTREE = 256       # shape detail: 256 good on a T4, 384 finer (more memory and time)
MAX_FACES = 40000  # the mesh is reduced to this many faces (keeps the file small for the walkthrough)

import torch
assert torch.cuda.is_available(), "No GPU. Session options > Accelerator > GPU T4 x2, then run again."
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install Hunyuan3D-2
The texture painter compiles two small CUDA extensions (a few minutes, only when `TEXTURE = True`).

The install changes numpy, so this cell **restarts the kernel by itself** the first time.
After the restart just **Run All** again: this cell sees the install is done and skips it.

In [ ]:
import os
DONE = "/tmp/.hunyuan_installed_v2" + ("_tex" if TEXTURE else "")
if os.path.exists(DONE):
    print("already installed, skipping")
else:
    if not os.path.isdir("/kaggle/temp/Hunyuan3D-2"):
        !git clone --depth 1 https://github.com/Tencent/Hunyuan3D-2 /kaggle/temp/Hunyuan3D-2
    %cd /kaggle/temp/Hunyuan3D-2
    !pip install -q -r requirements.txt
    !pip install -q -e .
    # Hunyuan3D-2 does not pin these; transformers 5 renamed the image encoder's weights, so it cannot load them
    !pip install -q "transformers==4.57.6" "diffusers==0.35.2" "huggingface-hub<1.0"
    !pip install -q fastapi uvicorn python-multipart pyngrok nest_asyncio
    if TEXTURE:
        %cd /kaggle/temp/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
        !python setup.py install > /dev/null
        %cd /kaggle/temp/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer
        !python setup.py install > /dev/null
    %cd /kaggle/working
    open(DONE, "w").close()
    print("installed. Restarting the kernel now: then Run All again.")
    os.kill(os.getpid(), 9)          # a clean restart, so the new numpy is the one that loads

## 3. Load the models

In [ ]:
import sys
sys.path.insert(0, "/kaggle/temp/Hunyuan3D-2")
import rembg as _rb
from hy3dgen.shapegen import (Hunyuan3DDiTFlowMatchingPipeline, FaceReducer, FloaterRemover,
                              DegenerateFaceRemover)

shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-2mini", subfolder="hunyuan3d-dit-v2-mini", variant="fp16")
_rb_session = _rb.new_session("u2net")      # small background remover (the default BRIA model fills the RAM)
rembg = lambda im: _rb.remove(im, session=_rb_session)

paint_pipe = None
if TEXTURE:
    try:
        from hy3dgen.texgen import Hunyuan3DPaintPipeline
        paint_pipe = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")
        paint_pipe.enable_model_cpu_offload()      # keeps a 15 GB T4 from running out of memory
    except Exception as e:
        print(f"Texture painter not available ({type(e).__name__}: {e}). Models will be shape only.")
print("models loaded | texture:", paint_pipe is not None)

## 4. Photo → GLB

- The background is removed, so the photo should show **one** piece, fully visible.
- The model has no real size and may face any way: the project scales it, and the app's *"Turn it"* option fixes the direction.

In [ ]:
import io, time
from PIL import Image

MAX_IMAGE_PX = 1024


def photo_to_glb(image_bytes: bytes, steps: int = STEPS, octree_resolution: int = OCTREE,
                 strip_background: bool = True, texture: bool = True, seed: int = 1234) -> bytes:
    image = Image.open(io.BytesIO(image_bytes))
    image.thumbnail((MAX_IMAGE_PX, MAX_IMAGE_PX))
    image = rembg(image.convert("RGB")) if strip_background else image.convert("RGBA")
    try:
        with torch.no_grad():
            mesh = shape_pipe(image=image, num_inference_steps=steps, octree_resolution=octree_resolution,
                              num_chunks=20000, generator=torch.manual_seed(seed), output_type="trimesh")[0]
        mesh = FloaterRemover()(mesh)
        mesh = DegenerateFaceRemover()(mesh)
        mesh = FaceReducer()(mesh, max_facenum=MAX_FACES)
        if texture and paint_pipe is not None:
            mesh = paint_pipe(mesh, image=image)
        return mesh.export(file_type="glb")
    finally:
        torch.cuda.empty_cache()


# quick self-test on Hunyuan3D's own example photo
t0 = time.time()
with open("/kaggle/temp/Hunyuan3D-2/assets/demo.png", "rb") as f:
    test_glb = photo_to_glb(f.read())
assert test_glb[:4] == b"glTF"
print(f"self-test ok: {len(test_glb) / 1e6:.1f} MB in {time.time() - t0:.0f} s")

## 5A. Your photos → GLB files
Every photo under `/kaggle/input` (*Add Input → Upload*) becomes a `.glb` in `/kaggle/working`.
Download them from the Output panel and use them in the app: *Use my own furniture → …or a GLB model*.

In [ ]:
import glob, os, time      # imported again here: after a restart the earlier cells' imports are gone

photos = sorted(p for e in ("jpg", "jpeg", "png", "webp") for p in glob.glob(f"/kaggle/input/**/*.{e}", recursive=True))
print(f"{len(photos)} photo(s) found" if photos else "No photos: Add Input > Upload your furniture photos, then run this cell again.")
for path in photos:
    t0 = time.time()
    out = "/kaggle/working/" + os.path.splitext(os.path.basename(path))[0].replace(" ", "_") + ".glb"
    with open(path, "rb") as f, open(out, "wb") as g:
        g.write(photo_to_glb(f.read()))
    print(f"{out}: {os.path.getsize(out) / 1e6:.1f} MB in {time.time() - t0:.0f} s")

## 5B. Or run it as the project's image-to-3D server

Same contract as `cv_service/colab_server.ipynb`: `GET /health`, `POST /generate` (field `image`, header `X-API-Key`), returns a GLB.
The project's `mc_resolution` (default 192) is used as half the shape detail (`octree_resolution`).
The server keeps running in the background after the last cell finishes, until the session stops.

In [ ]:
import threading

from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from kaggle_secrets import UserSecretsClient



def secret(name):
    try:
        value = UserSecretsClient().get_secret(name)
    except Exception:           # missing, or not attached to this notebook
        value = None
    assert value, f"Add the secret {name} (Add-ons > Secrets) and tick it for this notebook, then run this cell again."
    return value


API_KEY = secret("SMARTSEMSAR_KEY")

app = FastAPI(title="Smart Semsar furniture 3D (diffusion)")
gpu_lock = threading.Lock()     # one generation at a time


def check_key(x_api_key):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Wrong or missing X-API-Key.")


@app.get("/health")
def health(x_api_key: str = Header(default="")):
    check_key(x_api_key)
    return {"status": "ok", "model": "Hunyuan3D-2mini (diffusion)", "texture": paint_pipe is not None,
            "gpu": torch.cuda.get_device_name(0)}


@app.post("/generate")
def generate(image: UploadFile = File(...), mc_resolution: int = Form(OCTREE // 2),
             strip_background: bool = Form(True), steps: int = Form(STEPS), texture: bool = Form(True),
             x_api_key: str = Header(default="")):
    check_key(x_api_key)
    if not 64 <= mc_resolution <= 320:
        raise HTTPException(status_code=422, detail="mc_resolution must be between 64 and 320.")
    if not 5 <= steps <= 100:
        raise HTTPException(status_code=422, detail="steps must be between 5 and 100.")
    data = image.file.read()
    try:
        with gpu_lock:
            glb = photo_to_glb(data, steps=steps, octree_resolution=2 * mc_resolution,
                               strip_background=strip_background, texture=texture)
    except Exception as e:      # bad image, out of memory, ...
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {e}")
    return Response(content=glb, media_type="model/gltf-binary")

In [ ]:
import uvicorn
from pyngrok import ngrok

NGROK_TOKEN = secret("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print("Put these two lines in the project's .env file:\n")
print(f"SMARTSEMSAR_COLAB_URL={public_url}")
print("SMARTSEMSAR_COLAB_KEY=<the SMARTSEMSAR_KEY secret you chose>\n")

# uvicorn.run() fails inside a notebook (its event loop is already running),
# so the server runs in its own thread and keeps going after this cell finishes
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
print("server running in the background (until the session stops)")

In [ ]:
# keeps the session busy so Kaggle does not stop it while the server works (stop it with the square button)
import time
while True: time.sleep(60)